# K00 · Rehearsal of the locked analysis on the development events (before the freeze)

Runs **exactly the analysis code of the locked test** (`damagesar_bench.locked.run_locked_analysis`, used by k01)
on features that already exist, with the **development events standing in for the confirmatory events**. No
confirmatory label is read. Its purpose is to catch any bug in the locked analysis before the plan is frozen.

**Pass criteria** (checked at the end):
* RQ1 is the only comparison labelled confirmatory; every other one is exploratory; the level is 0.975.
* The satellite ViT-L and ImageNet AUROCs per event equal the recorded development values (p03, p04) within 0.005.
* RQ1's estimate equals the mean of the recorded per-event differences (+0.102).
* The supporting evidence (per-event paired tile-bootstrap, one-sided sign-flip) and RQ3 (C3) run.

Rehearsal-only shortcuts (not in k01): the satellite ViT-L's 10 m ring features are not computed here, so the
RQ4 ring comparison uses the outline features relabelled as ring (its number is meaningless); C3's source is
the other stand-in events plus any secondary event in the features (`c3_source_roles`), because the development
events play the confirmatory role. In k01 the source is the frozen one: development + secondary.

* **Before running:** Runtime → Change runtime type → **CPU**. Secret: `GH_TOKEN`. Needs p02, p03 and p04 features.
* **Time:** about 20–30 minutes. No compute units on a CPU runtime.

In [ ]:
NOTEBOOK_NAME = "k00_locked_rehearsal"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# Inputs: features from p02 (DINOv3 ViT-B), p03 (ImageNet, image-free) and p04 (satellite ViT-L)
import glob, os
from dataclasses import replace
from datetime import date
import pandas as pd
from damagesar_bench.config import load_events
P02 = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
P03 = sorted(glob.glob(f"{PROJECT}/02_models/features/*_controls_imagenet"))[-1]
P04 = sorted(glob.glob(f"{PROJECT}/02_models/features/*_more_encoders"))[-1]
real = load_events("/content/damagesar-bench/configs/events.yaml")
# Rehearsal config: development events play the confirmatory role; the real confirmatory events are left out
dev = real.keys_with_role("development")
ev = {k: replace(e, role="confirmatory") if k in dev else e
      for k, e in real.events.items() if e.role != "confirmatory"}
REH = replace(real, freeze_date=date(2020, 1, 1), events=ev)
print("stand-in confirmatory events:", REH.keys_with_role("confirmatory"))

In [ ]:
# Copy the features with the rehearsal roles (only the settings k01 uses)
os.makedirs("/content/k00", exist_ok=True)
def restage(path, keep, out, relabel=None):
    f = pd.read_parquet(path)
    f = f[[(e, p) in keep for e, p in zip(f["encoder"], f["pooling"])]].copy()
    f = f[f["event"].isin(REH.events)]
    f["role"] = f["event"].map(lambda e: REH.events[e].role)
    f["labels_locked"] = False
    if relabel:
        f["pooling"] = relabel
    f.to_parquet(out, index=False)
    print(out, f.groupby(["encoder", "pooling"]).size().to_dict())
    return out
FEATS = [
    restage(P02, {("dinov3_vitb16@native", "outline")}, "/content/k00/vitb16.parquet"),
    restage(f"{P03}/imagenet.parquet", {("imagenet_vitb16@native", "outline")}, "/content/k00/imagenet.parquet"),
    restage(f"{P03}/imagefree.parquet", {("imagefree", "geometry")}, "/content/k00/imagefree.parquet"),
    restage(f"{P04}/dinov3_vitl16_sat.parquet", {("dinov3_vitl16_sat@native", "outline")}, "/content/k00/sat.parquet"),
    restage(f"{P04}/dinov3_vitl16_sat.parquet", {("dinov3_vitl16_sat@native", "outline")},
            "/content/k00/sat_ring_standin.parquet", relabel="ring10m"),
]

In [ ]:
# The locked analysis, exactly as k01 calls it (RQ1 = satellite ViT-L, Gate 1 not passed, level 0.975)
from damagesar_bench.locked import run_locked_analysis
R = run_locked_analysis(FEATS, REH, RUN, rq1_encoder="dinov3_vitl16_sat", run_rq2=False, level=0.975,
                        c3_source_roles=("confirmatory", "secondary"))  # stand-ins = the development events
c = R["comparisons"]
print(c[["label", "strength", "n_events", "level", "estimate", "lower", "upper", "verdict", "sign_flip_p"]].round(4).to_string(index=False))
display(R["per_event"].round(4)); print(R["supporting"])
display(R["c3"][["event", "true_prevalence", "em", "bbse", "abstain", "estimate"]].round(3))

In [ ]:
# Pass criteria
import json
m = pd.read_csv(f"{R['bench_run']}/event_metrics.csv").set_index(["encoder", "event"])["auroc"]
want = {"dinov3_vitl16_sat@native": [0.742, 0.827, 0.755, 0.856, 0.646],
        "imagenet_vitb16@native": [0.465, 0.781, 0.716, 0.755, 0.600]}
evs = ["beirut", "congo", "hawaii", "la_palma", "libya"]
checks = {}
checks["only RQ1 confirmatory"] = list(c.loc[c["strength"] == "confirmatory", "label"]) == ["RQ1 DINOv3 vs ImageNet (outline)"]
checks["level 0.975"] = bool((c["level"] == 0.975).all())
checks["AUROCs match p03/p04"] = all(abs(m[(enc, e)] - v) <= 0.005 for enc, vals in want.items() for e, v in zip(evs, vals))
rq1 = c.set_index("label").loc["RQ1 DINOv3 vs ImageNet (outline)"]
checks["RQ1 estimate = +0.102"] = abs(rq1["estimate"] - 0.1018) <= 0.005
checks["supporting ran on 5 events"] = R["supporting"].n_events == 5
checks["C3 ran on 5 events"] = len(R["c3"]) == 5
for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)
json.dump({k: bool(v) for k, v in checks.items()}, open(f"{RUN}/k00_checks.json", "w"), indent=1)
print("\nREHEARSAL", "PASSED" if all(checks.values()) else "FAILED")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/checks/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))